In [ ]:
cd(joinpath(@__DIR__, ".."))   # run from the repository root

In [ ]:
using VoronoiMeshes, DelaunayTriangulation, NCDatasets, LinearAlgebra, JLD2
include("src/juliaDEC.jl");
@load "lsq_matrices.jld2" matrices_regular matrices_irregular

In [ ]:
error_lap_diagonal_linf = []
error_lap_lsq1_linf = []
error_lap_lsq2_linf = []
error_lap_diagonal_l2 = []
error_lap_lsq1_l2 = []
error_lap_lsq2_l2 = []

order_lap_diagonal_linf = []
order_lap_lsq1_linf = []
order_lap_lsq2_linf = []
order_lap_diagonal_l2 = []
order_lap_lsq1_l2 = []
order_lap_lsq2_l2 = []
edge_length_list = []
for i = 1:7
    mesh = VoronoiMesh("meshes/mesh" * string(i) * "_regular.nc")
    println("-----------------------------------------------------------")
    println("vertices = ", mesh.vertices.n, "    cells = ", mesh.cells.n, "     edges = ", mesh.edges.n)
    xp = mesh.x_period
    yp = mesh.y_period
    u = x -> sin(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp)
    true_laplacian = x -> -64*(pi^2)*sin(8*pi*x[1]/xp) * sin(8*pi*x[2]/yp)*(xp^(-2) + yp^(-2))

    u_cochain = Cochain(u.(mesh.cells.position), 0, true)
    lap_cochain = Cochain(true_laplacian.(mesh.cells.position), 0, true)
    f = lap_cochain.values
    F = sum(mesh.cells.area .* f)
    sum_areas = sum(mesh.cells.area)
    f .-= F/sum_areas

    d0 = d0_matrix_primal(mesh)
    d1 = d1_matrix_dual(mesh)
    star1_diagonal = star(mesh, 1, true)
    star1_lsq1 = matrices_regular[i]["lsq_linear"]
    star1_lsq2 = matrices_regular[i]["lsq_quadratic"]
    #star1_lsq1 = 0.5*(star1_lsq1 + star1_lsq1')
    #star1_lsq2 = 0.5*(star1_lsq2 + star1_lsq2')
    star2 = star(mesh, 2, false)

    A_diagonal = star2 * d1 * star1_diagonal * d0
    A_lsq1 = star2 * d1 * star1_lsq1 * d0
    A_lsq2 = star2 * d1 * star1_lsq2 * d0
    
    u_lap_diagonal = A_diagonal \ f
    u_lap_lsq1 = A_lsq1 \ f
    u_lap_lsq2 = A_lsq2 \ f
    u_lap_diagonal .-= sum(u_lap_diagonal .* mesh.cells.area)/sum_areas
    u_lap_lsq1 .-= sum(u_lap_lsq1 .* mesh.cells.area)/sum_areas
    u_lap_lsq2 .-= sum(u_lap_lsq2 .* mesh.cells.area)/sum_areas
    u_lap_diagonal = Cochain(u_lap_diagonal, 0, true)
    u_lap_lsq1 = Cochain(u_lap_lsq1, 0, true)
    u_lap_lsq2 = Cochain(u_lap_lsq2, 0, true)
    
    push!(error_lap_diagonal_l2, norm(u_lap_diagonal.values - u_cochain.values)*maximum(sqrt.(mesh.cells.area)))
    push!(error_lap_lsq1_l2, norm(u_lap_lsq1.values - u_cochain.values)*maximum(sqrt.(mesh.cells.area)))
    push!(error_lap_lsq2_l2, norm(u_lap_lsq2.values - u_cochain.values)*maximum(sqrt.(mesh.cells.area)))
    push!(error_lap_diagonal_linf, maximum(abs.(u_lap_diagonal.values - u_cochain.values)))
    push!(error_lap_lsq1_linf, maximum(abs.(u_lap_lsq1.values - u_cochain.values)))
    push!(error_lap_lsq2_linf, maximum(abs.(u_lap_lsq2.values - u_cochain.values)))
    push!(edge_length_list, maximum(sqrt.(mesh.cells.area)))
    println("Error Linf Diagonal = ", error_lap_diagonal_linf[end])
    println("Error Linf LSQ1 = ", error_lap_lsq1_linf[end])
    println("Error Linf LSQ2 = ", error_lap_lsq2_linf[end])
    println("Error L2 Diagonal = ", error_lap_diagonal_l2[end])
    println("Error L2 LSQ1 = ", error_lap_lsq1_l2[end])
    println("Error L2 LSQ2 = ", error_lap_lsq2_l2[end])
    if i > 1
        println("i = ", i)
        push!(order_lap_diagonal_linf, log(error_lap_diagonal_linf[end]/error_lap_diagonal_linf[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq1_linf, log(error_lap_lsq1_linf[end]/error_lap_lsq1_linf[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq2_linf, log(error_lap_lsq2_linf[end]/error_lap_lsq2_linf[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_diagonal_l2, log(error_lap_diagonal_l2[end]/error_lap_diagonal_l2[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq1_l2, log(error_lap_lsq1_l2[end]/error_lap_lsq1_l2[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq2_l2, log(error_lap_lsq2_l2[end]/error_lap_lsq2_l2[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
    end
end

println("order_lap_diagonal_Linf = ", order_lap_diagonal_linf)
println("order_lap_lsq1_Linf = ", order_lap_lsq1_linf)
println("order_lap_lsq2_Linf = ", order_lap_lsq2_linf)
println("order_lap_diagonal_l2 = ", order_lap_diagonal_l2)
println("order_lap_lsq1_l2 = ", order_lap_lsq1_l2)
println("order_lap_lsq2_l2 = ", order_lap_lsq2_l2)


In [ ]:
fig = Figure()
ax = Axis(fig[1,1])
plot_cochain!(ax, u_lap, mesh)
CairoMakie.save("laplacian_cochain.png", fig)


In [ ]:
error_lap_diagonal_linf = []
error_lap_lsq1_linf = []
error_lap_lsq2_linf = []
error_lap_diagonal_l2 = []
error_lap_lsq1_l2 = []
error_lap_lsq2_l2 = []

order_lap_diagonal_linf = []
order_lap_lsq1_linf = []
order_lap_lsq2_linf = []
order_lap_diagonal_l2 = []
order_lap_lsq1_l2 = []
order_lap_lsq2_l2 = []
edge_length_list = []
for i = 1:7
    mesh = VoronoiMesh("meshes/mesh" * string(i) * ".nc")
    println("-----------------------------------------------------------")
    println("vertices = ", mesh.vertices.n, "    cells = ", mesh.cells.n, "     edges = ", mesh.edges.n)
    xp = mesh.x_period
    yp = mesh.y_period
    u = x -> cos(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp)
    true_laplacian = x -> -64*(pi^2)*cos(8*pi*x[1]/xp) * cos(8*pi*x[2]/yp)*(xp^(-2) + yp^(-2))

    u_cochain = Cochain(u.(mesh.cells.position), 0, true)
    lap_cochain = Cochain(true_laplacian.(mesh.cells.position), 0, true)
    f = lap_cochain.values
    F = sum(mesh.cells.area .* f)
    sum_areas = sum(mesh.cells.area)
    f .-= F/sum_areas

    d0 = d0_matrix_primal(mesh)
    d1 = d1_matrix_dual(mesh)
    star1_diagonal = star(mesh, 1, true)
    star1_lsq1 = matrices_irregular[i]["lsq_linear"]
    star1_lsq2 = matrices_irregular[i]["lsq_quadratic"]
    #star1_lsq1 = 0.5*(star1_lsq1 + star1_lsq1')
    #star1_lsq2 = 0.5*(star1_lsq2 + star1_lsq2')
    star2 = star(mesh, 2, false)

    A_diagonal = star2 * d1 * star1_diagonal * d0
    A_lsq1 = star2 * d1 * star1_lsq1 * d0
    A_lsq2 = star2 * d1 * star1_lsq2 * d0
    
    u_lap_diagonal = A_diagonal \ f
    u_lap_lsq1 = A_lsq1 \ f
    u_lap_lsq2 = A_lsq2 \ f
    u_lap_diagonal .-= sum(u_lap_diagonal .* mesh.cells.area)/sum_areas
    u_lap_lsq1 .-= sum(u_lap_lsq1 .* mesh.cells.area)/sum_areas
    u_lap_lsq2 .-= sum(u_lap_lsq2 .* mesh.cells.area)/sum_areas
    u_lap_diagonal = Cochain(u_lap_diagonal, 0, true)
    u_lap_lsq1 = Cochain(u_lap_lsq1, 0, true)
    u_lap_lsq2 = Cochain(u_lap_lsq2, 0, true)
    
    push!(error_lap_diagonal_l2, norm(u_lap_diagonal.values - u_cochain.values)*maximum(sqrt.(mesh.cells.area)))
    push!(error_lap_lsq1_l2, norm(u_lap_lsq1.values - u_cochain.values)*maximum(sqrt.(mesh.cells.area)))
    push!(error_lap_lsq2_l2, norm(u_lap_lsq2.values - u_cochain.values)*maximum(sqrt.(mesh.cells.area)))
    push!(error_lap_diagonal_linf, maximum(abs.(u_lap_diagonal.values - u_cochain.values)))
    push!(error_lap_lsq1_linf, maximum(abs.(u_lap_lsq1.values - u_cochain.values)))
    push!(error_lap_lsq2_linf, maximum(abs.(u_lap_lsq2.values - u_cochain.values)))
    push!(edge_length_list, maximum(sqrt.(mesh.cells.area)))
    println("Error Linf Diagonal = ", error_lap_diagonal_linf[end])
    println("Error Linf LSQ1 = ", error_lap_lsq1_linf[end])
    println("Error Linf LSQ2 = ", error_lap_lsq2_linf[end])
    println("Error L2 Diagonal = ", error_lap_diagonal_l2[end])
    println("Error L2 LSQ1 = ", error_lap_lsq1_l2[end])
    println("Error L2 LSQ2 = ", error_lap_lsq2_l2[end])
    if i > 1
        println("i = ", i)
        push!(order_lap_diagonal_linf, log(error_lap_diagonal_linf[end]/error_lap_diagonal_linf[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq1_linf, log(error_lap_lsq1_linf[end]/error_lap_lsq1_linf[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq2_linf, log(error_lap_lsq2_linf[end]/error_lap_lsq2_linf[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_diagonal_l2, log(error_lap_diagonal_l2[end]/error_lap_diagonal_l2[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq1_l2, log(error_lap_lsq1_l2[end]/error_lap_lsq1_l2[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
        push!(order_lap_lsq2_l2, log(error_lap_lsq2_l2[end]/error_lap_lsq2_l2[end-1])/log(edge_length_list[end]/edge_length_list[end-1]))
    end
end

println("order_lap_diagonal_Linf = ", order_lap_diagonal_linf)
println("order_lap_lsq1_Linf = ", order_lap_lsq1_linf)
println("order_lap_lsq2_Linf = ", order_lap_lsq2_linf)
println("order_lap_diagonal_l2 = ", order_lap_diagonal_l2)
println("order_lap_lsq1_l2 = ", order_lap_lsq1_l2)
println("order_lap_lsq2_l2 = ", order_lap_lsq2_l2)

In [ ]:
mesh = VoronoiMesh("meshes/mesh4_regular.nc")
fig = Figure()
ax = Axis(fig[1,1])
plotmesh!(ax, mesh)
fig

In [ ]:
mesh = VoronoiMesh("meshes/mesh4.nc")
fig = Figure()
ax = Axis(fig[1,1])
plotmesh!(ax, mesh)
fig

In [ ]:
mesh = VoronoiMesh("meshes/mesh3_regular.nc")
d0 = d0_matrix_primal(mesh)
d1 = d1_matrix_dual(mesh)
star1_diagonal = star(mesh, 1, true)
#star1_lsq1 = hodge_star_lsq_matrix(mesh, true, true)
#star1_lsq2 = hodge_star_lsq_matrix(mesh, true, false)

In [ ]:
using LinearAlgebra
G = d1 * star1_diagonal  * d0
0 in [abs(sum(abs.(G[i, :])) - 2*abs(G[i,i])) < 1e-10 for i in axes(G, 1)]


In [ ]:
all(abs(sum(G[i, :]) - 2*G[i,i]) <= 1e-10 for i in axes(G, 1))

In [ ]:
[sum(abs.(G[i, :])) <= 2*abs(G[i,i])  for i in axes(G, 1)]